# combining chromosomal PON coverage and SNP data into combined data
+ mappability and GCratio is included for GC normalization and possible filtering
+ mappability and GC will not be saved in output files*

In [ ]:
# get the code
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# import package functions (main: sys.path.append('../code'); CNV_raw imports were unused here)
import mycnv

# paths and parameters: ../config/config_dev.yaml via dev_config.py (main: hard-coded Dropbox paths)
from dev_config import load_dev
dev = load_dev()
# PON outputs go to <out_root>/PON/, never into the cached PON (main wrote into PON_path)
pon_out = dev["pon_out"]
fig_path = os.path.join(pon_out, "figures")
os.makedirs(fig_path, exist_ok=True)
print(f"mycnv {mycnv.__version__} ({os.path.dirname(mycnv.__file__)})")
print(f"PON {dev['pon_dir']} -> {pon_out}")

### get the config

In [ ]:
CNVconfig = dev["CNVconfig"]
pon_dir = dev["pon_dir"]                       # reads cov/{chrom}.cov.gz and snp/{chrom}.snp.gz
chroms = dev["pon_build"].get("combine_chroms") or [f"chr{c + 1}" for c in range(22)] + ['chrX']
std_factor = CNVconfig["PONcoverage"]["stdFactor"]
CNVconfig

## Run the code

In [ ]:
# changes vs main (CHANGELOG.md): the filter stats are recomputed after the outlier removal and the full tables keep
# the unmodified coverage (legacy=True reproduces main); the tables come back as a dict and are written by write_pon
from mycnv.combine import make_pon_coverage, make_pon_snp, write_pon
tables = make_pon_coverage(pon_dir, std_factor=std_factor, chroms=chroms)
snp_df = make_pon_snp(pon_dir, chroms=chroms)
pon_files = write_pon(pon_out, tables, snp_df)   # main: save=True wrote into PON_path/CNV/
cov_df, filter_df = tables["full"], tables["filter"]

## Step by step

### PON coverage
+ combine chroms
+ normalize GCratio
+ adjust for male chrX coverage
+ filter outliers

In [ ]:
from mycnv.combine import combine_pon_coverage   # was an inline copy of combineCNV.combine_PON_coverage

In [ ]:
cov_df = combine_pon_coverage(pon_dir, chroms=chroms)
# look at general stats
cov_df.groupby("Chr").agg({"Cov1": "mean", "Cov2": "mean", "Pos":['count', 'min', 'max']}).sort_values("Chr")

### B) normalize GC ratio

###  Visualize GC bias
+ distribution of mean

In [ ]:
from mycnv.plot import make_GC_plot   # was an inline copy of code/plot.py:make_GC_plot

In [ ]:
from mycnv.plot import make_GC_plot
fig, _ = make_GC_plot(cov_df)
fig.savefig(os.path.join(fig_path, "GC_PON.jpeg"))

### visualize GC ratio distribution

In [ ]:
# main used normGC_cov_df here before defining it (cell order of an interactive session); compute it first
from mycnv.combine import normalize_gc
normGC_cov_df = normalize_gc(cov_df)
fig, _ = make_GC_plot(normGC_cov_df, agg="count")
fig.savefig(os.path.join(fig_path, "GC_PON_count.jpeg"))

### normalize the coverage to 100

In [ ]:
# canonical code (was an inline copy of combineCNV.normalize_GC_col/normalize_GC/normalize): mycnv.combine
from mycnv.combine import normalize_gc_col, normalize_gc, normalize

### visualize normalization

In [ ]:
normGC_cov_df = normalize_gc(cov_df)
norm_cov_df = normalize(cov_df)
del cov_df
fig, _ = make_GC_plot(normGC_cov_df)
fig.savefig(os.path.join(fig_path, "GC_PON_GCnorm.jpeg"))
fig, _ = make_GC_plot(norm_cov_df)
fig.savefig(os.path.join(fig_path, "GC_PON_norm.jpeg"))

### raise X-coverage for male samples
+ consense X coverage is required

In [ ]:
# main used an undefined cov_agg here; it is the per-sample mean
cov_agg = {col: "mean" for col in normGC_cov_df.columns if col.startswith("Cov")}
normGC_cov_df.groupby("Chr").agg(cov_agg).loc["chrX"]

In [ ]:
# canonical code (was an inline copy of combineCNV.amazonize): mycnv.combine.amazonize
# change vs main: one table in, one table out (main took *cov_dfs and returned a list, which broke the next cell)
from mycnv.combine import amazonize

In [ ]:
normX_df = amazonize(norm_cov_df)
del norm_cov_df
normX_df.groupby("Chr").agg(cov_agg).loc["chrX"]

In [ ]:
normX_df

### compute the mean of all the coverages

In [ ]:
# canonical code (was an inline copy of combineCNV.compute_stats): one table in, one table out (main: *dfs -> list)
from mycnv.combine import compute_stats

In [ ]:
mean_df = compute_stats(normX_df)
del normX_df
mean_df

### filter the coverage
+ for filtering all outliers are removed that stray from the local mean above max_mean_std
+ other filtering steps per position should be done afterwards during rolling window stuff:
+ `filter_df = df.query('meanCov > @mincov and std < @max_mean_std')`


In [ ]:
mean_df['PONcov_std'].max()
mean_df['PONcov_std'].mean()

### remove outliers in order to reduce noise

In [ ]:
# canonical code (was an inline copy of combineCNV.remove_outliers): one table in, one table out (main: *dfs -> list)
from mycnv.combine import remove_outliers

In [ ]:
filter_df = compute_stats(remove_outliers(mean_df, std_factor=2))
del mean_df
filter_df

In [ ]:
filter_df['PONcov_std'].max()
filter_df['PONcov_std'].mean()

### the master function

In [ ]:
# canonical code (was an inline copy of combineCNV.make_PON_coverage): mycnv.combine.make_pon_coverage returns the four
# tables as a dict (full, fullGC, filter, filterGC); mycnv.combine.write_pon writes them; CLI: mycnv pon-combine
from mycnv.combine import make_pon_coverage

In [ ]:
tables = make_pon_coverage(pon_dir, std_factor=std_factor, chroms=chroms)
cov_df, filter_df = tables["full"], tables["filter"]
filter_df[:10]

### PON SNP
+ combine chroms

In [ ]:
from mycnv.combine import make_pon_snp   # was an inline copy of combineCNV.make_PON_snp

In [ ]:
snp_df = make_pon_snp(pon_dir, chroms=chroms)
snp_df